In [ ]:
import os
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
import torch_geometric.transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
from dataset.cell_motion import CellMotion, CellMotionRollout
from model.simulator import Simulator
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

dataset_dir = os.path.join(os.getcwd(), 'dataset')
sequence_length = 20
train_rollout_steps = 6
val_rollout_steps = 8
batch_size = 16
max_epochs = 400
hidden_size = 256
message_passing_num = 15
learning_rate = 5e-4
rollout_weight = 0.4
teacher_forcing_start = 0.9
teacher_forcing_min = 0.3
teacher_forcing_decay = 0.995
max_grad_norm = 1.0

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = os.path.join('checkpoint_long', timestamp)
os.makedirs(save_dir, exist_ok=True)


In [ ]:
def compute_velocity_features(flat_history: torch.Tensor, sequence_len: int):
    time_feature = flat_history[:, -1:]
    pos_flat = flat_history[:, :-1]
    num_nodes = pos_flat.size(0)
    pos_seq = pos_flat.view(num_nodes, sequence_len, 2)
    velocities = pos_seq[:, 1:, :] - pos_seq[:, :-1, :]
    zeros = torch.zeros(num_nodes, 1, 2, device=flat_history.device)
    velocities = torch.cat([zeros, velocities], dim=1).view(num_nodes, -1)
    return torch.cat([velocities, time_feature], dim=1)

def rollout_loss(model: Simulator, graph: Data, rollout_steps: int, teacher_forcing_prob: float):
    num_nodes = graph.pos.size(0)
    history = graph.x[:, :sequence_length * 2].view(num_nodes, sequence_length, 2)
    time_col = graph.x[:, -1:]
    future = getattr(graph, 'future_pos', None)
    if future is None:
        future = graph.y.unsqueeze(0)
    max_steps = min(rollout_steps, future.shape[0])
    current_pos = graph.pos
    last_disp = current_pos - history[:, -1]
    losses = []
    for step in range(max_steps):
        vel_input = compute_velocity_features(torch.cat([history.reshape(num_nodes, -1), time_col], dim=1), sequence_length)
        rollout_graph = Data(
            x=vel_input,
            pos=current_pos,
            edge_index=graph.edge_index,
            edge_attr=getattr(graph, 'edge_attr', None)
        )
        pred_norm = model.model(rollout_graph)
        target_pos = future[step]
        target_vel = target_pos - current_pos
        target_norm = model._output_normalizer(target_vel, model.training)
        losses.append(torch.mean((pred_norm - target_norm) ** 2))
        pred_vel = model._output_normalizer.inverse(pred_norm)
        blended_vel = 0.8 * pred_vel + 0.2 * last_disp
        next_pos_pred = current_pos + blended_vel
        use_teacher = torch.rand(1).item() < teacher_forcing_prob
        next_pos = target_pos if use_teacher else next_pos_pred.detach()
        history = torch.cat([history[:, 1:], next_pos.unsqueeze(1)], dim=1)
        time_col = time_col + 1.0
        last_disp = next_pos - current_pos
        current_pos = next_pos
    return torch.stack(losses).mean()

def one_step_loss(model: Simulator, graph: Data):
    vel_input = compute_velocity_features(graph.x, sequence_length)
    graph_one = graph.clone()
    graph_one.x = vel_input
    pred_norm = model.model(graph_one)
    target_vel = graph.y - graph.pos
    target_norm = model._output_normalizer(target_vel, model.training)
    return torch.mean((pred_norm - target_norm) ** 2)


In [ ]:
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

train_dataset = CellMotion(
    dataset_dir=dataset_dir,
    split='train_e0n300',
    max_epochs=max_epochs,
    sequence_length=sequence_length,
    predict_steps=train_rollout_steps
)
valid_dataset = CellMotionRollout(
    dataset_dir=dataset_dir,
    split='valid_e0n400',
    sequence_length=sequence_length,
    predict_steps=val_rollout_steps
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, num_workers=0)

simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=sequence_length * 2 + 1,
    edge_input_size=3,
    device=device,
    hidden_size=hidden_size,
    model_dir=save_dir
)

optimizer = torch.optim.Adam(simulator.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=30,
    min_lr=1e-6,
    verbose=True
)

teacher_forcing_prob = teacher_forcing_start
best_val = float('inf')
best_path = os.path.join(save_dir, 'best_checkpoint.pth')

for epoch in range(1, max_epochs + 1):
    simulator.train()
    train_losses = []
    for graph in train_loader:
        graph = transformer(graph)
        graph = graph.to(device)
        loss_main = one_step_loss(simulator, graph)
        loss_roll = rollout_loss(simulator, graph, train_rollout_steps, teacher_forcing_prob)
        loss = (1.0 - rollout_weight) * loss_main + rollout_weight * loss_roll
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(simulator.parameters(), max_grad_norm)
        optimizer.step()
        train_losses.append(loss.item())

    simulator.eval()
    val_losses = []
    with torch.no_grad():
        for graph in valid_loader:
            graph = transformer(graph)
            graph = graph.to(device)
            l_main = one_step_loss(simulator, graph)
            l_roll = rollout_loss(simulator, graph, val_rollout_steps, teacher_forcing_min)
            val_losses.append(((1.0 - rollout_weight) * l_main + rollout_weight * l_roll).item())

    val_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else 0.0
    scheduler.step(val_loss)
    teacher_forcing_prob = max(teacher_forcing_min, teacher_forcing_prob * teacher_forcing_decay)

    if val_loss < best_val:
        best_val = val_loss
        simulator.save_checkpoint(best_path)

    print(f'Epoch {epoch:04d} | train {np.mean(train_losses):.4e} | val {val_loss:.4e} | tf_prob {teacher_forcing_prob:.3f}')

print(f'Best validation loss: {best_val:.4e}')
print(f'Checkpoint saved to {best_path}')
